In [1]:
import pandas as pd
import numpy as np

import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Configuration dictionary for thresholds and priorities
config = {
    'visibility_threshold': 0.5,
    'reference_priority': ['mid_hip', 'mid_shoulder', 'dataset_mean'],
    'scale_priority': ['shoulder_width', 'hip_width', 'max_pairwise']
}

In [2]:
def get_landmarks(df):
    """
    Extract unique landmark names from DataFrame columns.
    Assumes columns are named as 'landmark_x', 'landmark_y', 'landmark_z', 'landmark_v'.
    """
    landmarks = set()
    for col in df.columns:
        if col.endswith('_x'):
            landmarks.add(col[:-2])  # Remove '_x' to get landmark name
    return list(landmarks)

In [3]:
def determine_reference_strategy(df, config):
    """
    Determine the reference point strategy for the entire dataset based on priority and visibility.
    Ensures consistency across all rows by choosing one strategy.
    """
    landmarks = get_landmarks(df)
    thresh = config['visibility_threshold']
    for strategy in config['reference_priority'][:-1]:  # Exclude dataset_mean as it's fallback
        if strategy == 'mid_hip':
            lm1, lm2 = 'left_hip', 'right_hip'
        elif strategy == 'mid_shoulder':
            lm1, lm2 = 'left_shoulder', 'right_shoulder'
        if lm1 in landmarks and lm2 in landmarks:
            mean_v = (df[f'{lm1}_v'].mean() + df[f'{lm2}_v'].mean()) / 2
            if mean_v > thresh:
                logging.info(f"Using {strategy} as reference strategy.")
                return strategy
    logging.info("Using dataset_mean as reference strategy (fallback).")
    return 'dataset_mean'

In [4]:
def compute_global_reference(df, strategy, config):
    """
    Compute the global reference point based on the chosen strategy.
    For mid_hip/mid_shoulder: average of mid-points from rows where both landmarks are visible.
    For dataset_mean: average position of all visible landmarks across the dataset.
    This ensures temporal consistency.
    """
    thresh = config['visibility_threshold']
    landmarks = get_landmarks(df)
    if strategy in ['mid_hip', 'mid_shoulder']:
        if strategy == 'mid_hip':
            lm1, lm2 = 'left_hip', 'right_hip'
        else:
            lm1, lm2 = 'left_shoulder', 'right_shoulder'
        mask = (df[f'{lm1}_v'] > thresh) & (df[f'{lm2}_v'] > thresh)
        if mask.any():
            mid_x = (df.loc[mask, f'{lm1}_x'] + df.loc[mask, f'{lm2}_x']) / 2
            mid_y = (df.loc[mask, f'{lm1}_y'] + df.loc[mask, f'{lm2}_y']) / 2
            mid_z = (df.loc[mask, f'{lm1}_z'] + df.loc[mask, f'{lm2}_z']) / 2
            ref_x = mid_x.mean()
            ref_y = mid_y.mean()
            ref_z = mid_z.mean()
            return ref_x, ref_y, ref_z
        else:
            logging.warning(f"No rows with both {lm1} and {lm2} visible, falling back to dataset_mean.")
            return compute_global_reference(df, 'dataset_mean', config)
    else:  # dataset_mean
        all_x, all_y, all_z = [], [], []
        for lm in landmarks:
            mask = df[f'{lm}_v'] > thresh
            if mask.any():
                all_x.extend(df.loc[mask, f'{lm}_x'].values)
                all_y.extend(df.loc[mask, f'{lm}_y'].values)
                all_z.extend(df.loc[mask, f'{lm}_z'].values)
        if all_x:
            ref_x = np.mean(all_x)
            ref_y = np.mean(all_y)
            ref_z = np.mean(all_z)
            return ref_x, ref_y, ref_z
        else:
            logging.warning("No visible landmarks in dataset, using origin as reference.")
            return 0.0, 0.0, 0.0

In [5]:
def determine_scale_strategy(df, config):
    """
    Determine the scale normalization strategy for the entire dataset based on priority and visibility.
    Ensures consistency across all rows by choosing one method.
    """
    landmarks = get_landmarks(df)
    thresh = config['visibility_threshold']
    for strategy in config['scale_priority'][:-1]:  # Exclude max_pairwise as fallback
        if strategy == 'shoulder_width':
            lm1, lm2 = 'left_shoulder', 'right_shoulder'
        elif strategy == 'hip_width':
            lm1, lm2 = 'left_hip', 'right_hip'
        if lm1 in landmarks and lm2 in landmarks:
            mean_v = (df[f'{lm1}_v'].mean() + df[f'{lm2}_v'].mean()) / 2
            if mean_v > thresh:
                logging.info(f"Using {strategy} as scale strategy.")
                return strategy
    logging.info("Using max_pairwise as scale strategy (fallback).")
    return 'max_pairwise'

In [6]:
def compute_global_scale(df, strategy, config):
    """
    Compute the global scale factor based on the chosen strategy.
    For shoulder_width/hip_width: average width from rows where both landmarks are visible.
    For max_pairwise: maximum pairwise Euclidean distance across all visible landmarks in the dataset.
    This ensures scale invariance and consistency.
    """
    thresh = config['visibility_threshold']
    landmarks = get_landmarks(df)
    if strategy in ['shoulder_width', 'hip_width']:
        if strategy == 'shoulder_width':
            lm1, lm2 = 'left_shoulder', 'right_shoulder'
        else:
            lm1, lm2 = 'left_hip', 'right_hip'
        mask = (df[f'{lm1}_v'] > thresh) & (df[f'{lm2}_v'] > thresh)
        if mask.any():
            dists = np.sqrt(
                (df.loc[mask, f'{lm1}_x'] - df.loc[mask, f'{lm2}_x'])**2 +
                (df.loc[mask, f'{lm1}_y'] - df.loc[mask, f'{lm2}_y'])**2 +
                (df.loc[mask, f'{lm1}_z'] - df.loc[mask, f'{lm2}_z'])**2
            )
            scale = dists.mean()
            if scale <= 1e-6:
                logging.warning(f"Scale factor too small ({scale}), using 1.0 as safe default.")
                scale = 1.0
            return scale
        else:
            logging.warning(f"No rows with both {lm1} and {lm2} visible, falling back to max_pairwise.")
            return compute_global_scale(df, 'max_pairwise', config)
    else:  # max_pairwise
        visible_points = []
        for lm in landmarks:
            mask = df[f'{lm}_v'] > thresh
            if mask.any():
                points = df.loc[mask, [f'{lm}_x', f'{lm}_y', f'{lm}_z']].values
                visible_points.extend(points)
        if len(visible_points) >= 2:
            max_dist = 0.0
            for i in range(len(visible_points)):
                for j in range(i + 1, len(visible_points)):
                    dist = np.linalg.norm(visible_points[i] - visible_points[j])
                    if dist > max_dist:
                        max_dist = dist
            if max_dist <= 1e-6:
                logging.warning(f"Max distance too small ({max_dist}), using 1.0 as safe default.")
                max_dist = 1.0
            return max_dist
        else:
            logging.warning("Fewer than 2 visible points, using 1.0 as safe default.")
            return 1.0

In [7]:
def center_coordinates(row, ref, landmarks):
    """
    Center the coordinates by subtracting the global reference point from each landmark's x, y, z.
    Visibility v remains unchanged. This ensures translation invariance.
    """
    centered = {}
    for lm in landmarks:
        centered[f'{lm}_x'] = row[f'{lm}_x'] - ref[0]
        centered[f'{lm}_y'] = row[f'{lm}_y'] - ref[1]
        centered[f'{lm}_z'] = row[f'{lm}_z'] - ref[2]
        centered[f'{lm}_v'] = row[f'{lm}_v']
    return centered

In [8]:
def scale_coordinates(centered, scale, landmarks):
    """
    Scale the centered coordinates by dividing x, y, z by the global scale factor.
    Visibility v remains unchanged. This ensures scale invariance.
    """
    scaled = {}
    for lm in landmarks:
        scaled[f'{lm}_x_scaled'] = centered[f'{lm}_x'] / scale
        scaled[f'{lm}_y_scaled'] = centered[f'{lm}_y'] / scale
        scaled[f'{lm}_z_scaled'] = centered[f'{lm}_z'] / scale
        scaled[f'{lm}_v'] = centered[f'{lm}_v']
    return scaled

In [9]:
# Old compute_scale removed, now using global scale

In [10]:

def scale_coordinates(centered, scale, landmarks):
    """
    Scale the centered coordinates by dividing x, y, z by the scale factor.
    Visibility v remains unchanged.
    Returns a dict with scaled values.
    """
    scaled = {}
    for lm in landmarks:
        scaled[f'{lm}_x'] = centered[f'{lm}_x'] / scale
        scaled[f'{lm}_y'] = centered[f'{lm}_y'] / scale
        scaled[f'{lm}_z'] = centered[f'{lm}_z'] / scale
        scaled[f'{lm}_v'] = centered[f'{lm}_v']
    return scaled



In [11]:
def preprocess_pose_df(df, config=config):
    """
    Preprocess the pose DataFrame with consistent reference and scale strategies.
    Determines global reference and scale once, then applies to all rows for temporal consistency.
    Produces translation-invariant and scale-invariant features suitable for ML.
    Preserves the 'label' column unchanged.
    """
    landmarks = get_landmarks(df)
    ref_strategy = determine_reference_strategy(df, config)
    scale_strategy = determine_scale_strategy(df, config)
    global_ref = compute_global_reference(df, ref_strategy, config)
    global_scale = compute_global_scale(df, scale_strategy, config)
    
    processed_rows = []
    for idx, row in df.iterrows():
        centered = center_coordinates(row, global_ref, landmarks)
        scaled = scale_coordinates(centered, global_scale, landmarks)
        processed_rows.append(scaled)
    processed_df = pd.DataFrame(processed_rows)
    # Preserve the label column unchanged
    if 'label' in df.columns:
        processed_df['label'] = df['label'].values
    return processed_df

In [12]:
def preprocess_pose_csv(csv_path, output_path=None, config=config):
    """
    Load CSV, preprocess with consistent strategies, and optionally save.
    Returns the processed DataFrame with normalized, ML-ready features.
    """
    df = pd.read_csv(csv_path)
    processed_df = preprocess_pose_df(df, config)
    if output_path:
        processed_df.to_csv(output_path, index=False)
    return processed_df

In [13]:
if __name__ == "__main__":
    # Example usage
    input_csv = "train_augmented.csv"
    output_csv = f"scaled_{input_csv}"
    processed_df = preprocess_pose_csv(input_csv, output_csv, config)
    print(f"Processed data saved to {output_csv}")

INFO: Using mid_hip as reference strategy.
INFO: Using shoulder_width as scale strategy.
INFO: Using shoulder_width as scale strategy.


Processed data saved to scaled_train_augmented.csv
